# Week 9 Lab — Linear regression clinic

**HWRS 564a · Fall 2026**

Today is a full-period working session. You will derive, fit, diagnose,
validate, and interpret one simple linear model using a 63-year groundwater
record. The purpose is not to collect outputs from a function. It is to decide
which statements those outputs support.

## Learning objectives

By the end of the lab you can:

1. reproduce the OLS slope and intercept from centred sums;
2. explain how centring changes the intercept but not the slope;
3. calculate RSS, TSS, $R^2$, residual variance, slope standard error, and a
   confidence interval;
4. diagnose nonlinearity, unequal variance, non-normality, and serial correlation;
5. compare interpolation and forecasting validation; and
6. separate descriptive, inferential, predictive, and causal claims.

Run cells in order. Every `# YOUR TURN` cell has a following `# CHECK` cell.
Before leaving, restart the kernel and run all cells.


---

## Part 1 — Prepare a predictor with interpretable units

We will model depth to water as a linear function of elapsed time. Because
depth is measured downward, a positive slope means the water table becomes
deeper: a decline in hydraulic head.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists()
)
DATA = ROOT / "data"

levels = pd.read_csv(
    DATA / "tucson_water_levels.csv",
    dtype={"site_no": str},
    parse_dates=["date"],
)

one = (
    levels.loc[levels["site_no"] == "320824110593001"]
    .sort_values("date")
    .dropna(subset=["date", "depth_to_water_ft"])
    .copy()
)
one["years_elapsed"] = (one["date"] - one["date"].min()).dt.days / 365.25

print(f"{len(one)} measurements from {one['date'].min():%Y} to {one['date'].max():%Y}")
print(one[["date", "years_elapsed", "depth_to_water_ft"]].head())


Common pitfall: dropping missing values from $x$ and $y$ separately can destroy
their row-by-row pairing. Drop incomplete rows from both required columns in
one operation, as above.

### YOUR TURN 1 — Reproduce the OLS coefficients

Use the formulas derived Tuesday:

$$
b_1=rac{\sum_i(x_i-ar x)(y_i-ar y)}
          {\sum_i(x_i-ar x)^2},
\qquad b_0=ar y-b_1ar x.
$$


In [ ]:
x = one["years_elapsed"].to_numpy()
y = one["depth_to_water_ft"].to_numpy()
x_bar, y_bar = x.mean(), y.mean()

# YOUR TURN
sxx = ...
sxy = ...
manual_slope = ...
manual_intercept = ...


In [ ]:
# CHECK
np.testing.assert_allclose(sxx, 131428.3168427)
np.testing.assert_allclose(sxy, 273520.1171342)
np.testing.assert_allclose(manual_slope, 2.0811353573)
np.testing.assert_allclose(manual_intercept, 24.6629112752)
print(f"depth = {manual_intercept:.3f} + {manual_slope:.3f} × elapsed years")
print("Correct.")


Interpret the slope: the fitted depth increased by about 2.08 ft per year over
this record. Because depth increases downward, that is an average water-level
decline of 2.08 ft/yr.

The intercept is the fitted depth when elapsed time is zero—the first date in
the record. It is not a universal baseline.


---

## Part 2 — Centre the predictor to make the intercept useful

Changing the origin of $x$ should not change the slope. It changes which date
the intercept describes.

### YOUR TURN 2 — Put the intercept at 1970

Create elapsed years from 1 January 1970, fit the same response, and record the
new intercept. `fit_1970.intercept` estimates depth at the 1970 reference date.


In [ ]:
one["years_from_1970"] = (
    one["date"] - pd.Timestamp("1970-01-01")
).dt.days / 365.25

fit = stats.linregress(one["years_elapsed"], one["depth_to_water_ft"])

# YOUR TURN
fit_1970 = ...
slope_change = ...
depth_at_1970 = ...


In [ ]:
# CHECK
np.testing.assert_allclose(slope_change, 0.0, atol=1e-12)
np.testing.assert_allclose(depth_at_1970, 104.7175362453)
print(f"original intercept: {fit.intercept:.2f} ft at the first observation")
print(f"centred intercept: {depth_at_1970:.2f} ft at 1970-01-01")
print(f"slope in both fits: {fit_1970.slope:.3f} ft/yr")
print("Correct.")


Common pitfall: interpreting an intercept at $x=0$ when zero is far outside the
data or has no physical meaning. Centre the predictor at a meaningful date or
condition. This changes the intercept and its uncertainty, not the fitted line.


---

## Part 3 — Reconstruct fit and uncertainty statistics

First compute predictions and residuals. Here residual means **measured minus
predicted**.


In [ ]:
one["predicted"] = manual_intercept + manual_slope * one["years_elapsed"]
one["residual"] = one["depth_to_water_ft"] - one["predicted"]

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(one["date"], one["depth_to_water_ft"], "o", ms=2.5,
             color="#49595E", alpha=0.55, label="measured")
axes[0].plot(one["date"], one["predicted"], color="#AB0520", lw=2,
             label="OLS line")
axes[0].invert_yaxis()
axes[0].set_ylabel("depth to water (ft)")
axes[0].legend(frameon=False)

axes[1].plot(one["date"], one["residual"], "o", ms=2.5,
             color="#1E5288", alpha=0.65)
axes[1].axhline(0, color="k", lw=1)
axes[1].set_ylabel("residual (ft)")
axes[1].set_xlabel("date")
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


### YOUR TURN 3 — Calculate $R^2$, standard error, and the interval

Use

$$
R^2=1-\frac{\mathrm{RSS}}{\mathrm{TSS}},\quad
s^2=\frac{\mathrm{RSS}}{n-2},\quad
\mathrm{SE}(b_1)=\sqrt{\frac{s^2}{S_{xx}}}.
$$

Then form a 95% interval with the Student-$t$ critical value.


In [ ]:
residual = one["residual"].to_numpy()
n = len(one)
t_critical = stats.t.ppf(0.975, df=n - 2)

# YOUR TURN
rss = ...
tss = ...
r_squared = ...
residual_variance = ...
slope_se = ...
margin = ...
ci_low = ...
ci_high = ...


In [ ]:
# CHECK
np.testing.assert_allclose(rss, 64167.0382349)
np.testing.assert_allclose(tss, 633399.4249409)
np.testing.assert_allclose(r_squared, 0.8986941956)
np.testing.assert_allclose(residual_variance, 170.20434545)
np.testing.assert_allclose(slope_se, fit.stderr)
np.testing.assert_allclose([ci_low, ci_high], [2.01037575, 2.15189496])
print(f"R2 = {r_squared:.3f}")
print(f"slope = {manual_slope:.3f} ± {slope_se:.3f} ft/yr (ordinary SE)")
print(f"ordinary 95% CI: {ci_low:.3f} to {ci_high:.3f} ft/yr")
print("Correct.")


The calculation is correct. Whether the interval is scientifically credible is
a separate question. It assumes a linear mean, independent errors, and constant
error variance; exact small-sample $t$ inference also assumes normal errors.


---

## Part 4 — Diagnose the assumptions

The plots already show long runs above and below zero. Quantify that structure
and compare the residual spread in the lower and upper halves of the fitted
values.

### YOUR TURN 4 — Measure dependence and changing spread


In [ ]:
year = one["date"].dt.year

# YOUR TURN
by_decade = ...
lag1 = ...
low_spread = ...
high_spread = ...
spread_ratio = ...


In [ ]:
# CHECK
assert by_decade.idxmin() == 1950
assert by_decade.idxmax() == 1970
np.testing.assert_allclose(lag1, 0.986071125)
np.testing.assert_allclose(low_spread, 8.490762596)
np.testing.assert_allclose(high_spread, 15.351717640)
np.testing.assert_allclose(spread_ratio, 1.80804933)
print(by_decade.round(1).to_string())
print()
print(f"lag-1 residual correlation: {lag1:.3f}")
print(f"upper/lower fitted-range spread ratio: {spread_ratio:.2f}")
print("Correct.")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

axes[0].scatter(one["predicted"], one["residual"], s=14,
                color="#1E5288", alpha=0.6)
axes[0].axhline(0, color="k", lw=1)
axes[0].set_xlabel("fitted depth (ft)")
axes[0].set_ylabel("residual (ft)")
axes[0].set_title("Residual versus fitted")

axes[1].scatter(one["residual"].iloc[:-1], one["residual"].iloc[1:],
                s=14, color="#AB0520", alpha=0.6)
axes[1].set_xlabel("residual at observation i")
axes[1].set_ylabel("residual at observation i + 1")
axes[1].set_title(f"Lag plot, r = {lag1:.3f}")

stats.probplot(one["residual"], dist="norm", plot=axes[2])
axes[2].set_title("Normal Q–Q plot")

for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


Interpretation:

- Decade-long residual structure challenges a single linear mean.
- Lag-1 correlation of 0.986 strongly rejects independence.
- Residual spread is about 1.8 times larger in the upper fitted range,
  challenging constant variance.
- The Q–Q plot checks distributional shape, but normal-looking residuals would
  not repair dependence or nonlinearity.

The OLS line remains a descriptive line. The ordinary standard error and
confidence interval are much harder to defend.


---

## Part 5 — Validation changes with the prediction question

A random split asks whether the line can interpolate within an era represented
in training. A chronological split asks whether an early linear trend can
forecast a later era.

### YOUR TURN 5 — Compare both designs


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

X = one[["years_elapsed"]].to_numpy()
y = one["depth_to_water_ft"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=564
)
random_model = LinearRegression().fit(X_train, y_train)

# YOUR TURN
random_prediction = ...
random_mae = ...
random_rmse = ...
random_r2 = ...

time_mask = ...
time_model = ...
time_prediction = ...
time_mae = ...
time_rmse = ...
time_r2 = ...


In [ ]:
# CHECK
np.testing.assert_allclose(random_mae, 11.27230049)
np.testing.assert_allclose(random_rmse, 13.29259263)
np.testing.assert_allclose(random_r2, 0.89142489)
assert time_mask.sum() == 188 and (~time_mask).sum() == 191
np.testing.assert_allclose(time_mae, 15.39927091)
np.testing.assert_allclose(time_rmse, 19.34386821)
np.testing.assert_allclose(time_r2, -4.80587106)
print("                 MAE      RMSE      R2")
print(f"random split   {random_mae:6.1f}    {random_rmse:6.1f}   {random_r2:6.2f}")
print(f"time split     {time_mae:6.1f}    {time_rmse:6.1f}   {time_r2:6.2f}")
print("Correct.")


Common pitfalls:

- Test $R^2$ can be negative. It means predictions are worse than using the
  test-set mean as a baseline.
- Random splitting is not inherently wrong; it answers an interpolation
  question. It is wrong when presented as evidence of forecasting ability.
- MAE and RMSE retain the response units. RMSE reacts more strongly to large
  misses.


---

## Part 6 — Write only the claims the evidence supports

### YOUR TURN 6 — Classify the conclusions

Complete the four variables. Use `True` only when the analysis supports the
claim without an unaddressed assumption failure.


In [ ]:
# YOUR TURN
descriptive_rate = ...
ordinary_ci_is_trustworthy = ...
forecast_model_is_adequate = ...
causal_claim_is_supported = ...
primary_warning = ...


In [ ]:
# CHECK
np.testing.assert_allclose(descriptive_rate, 2.0811353573)
assert ordinary_ci_is_trustworthy is False
assert forecast_model_is_adequate is False
assert causal_claim_is_supported is False
assert "residual" in primary_warning.lower()
print(f"Descriptive rate: {descriptive_rate:.2f} ft/yr of increasing depth")
print(f"Trust ordinary CI without adjustment? {ordinary_ci_is_trustworthy}")
print(f"Adequate for forecasting the later era? {forecast_model_is_adequate}")
print(f"Does elapsed time establish causation? {causal_claim_is_supported}")
print(f"Primary warning: {primary_warning}")
print("Correct.")


Now write a four-sentence conclusion in your own words:

1. **Descriptive:** state the fitted rate, direction, units, site, and period.
2. **Inferential:** state that the narrow ordinary interval is conditional on
   assumptions that the residual diagnostics challenge.
3. **Predictive:** state what the chronological validation shows.
4. **Causal:** name at least one plausible process that elapsed time does not
   itself identify, such as pumping or recharge.

Avoid these phrases:

- “$R^2=0.90$, so the model is 90% accurate.”
- “The p-value proves groundwater decline.”
- “Time caused the water table to fall.”
- “The confidence interval contains 95% of future observations.”


---

## Before you leave

1. Restart the kernel and run all cells.
2. Resolve every failed check.
3. Save the notebook.
4. Confirm that your written interpretation separates descriptive,
   inferential, predictive, and causal statements.

## If you get stuck

- `linregress` returns `nan`: drop incomplete $x$–$y$ rows together.
- Your slope has units of ft/day: convert the datetime difference to years.
- Your residuals have the opposite sign: use measured minus predicted.
- Your standard error is close but not equal: residual variance uses $n-2$.
- Your random-split values change: use `random_state=564`.
- `Expected 2D array`: scikit-learn requires `X` shaped `(rows, predictors)`;
  use double brackets around the predictor column.
